# Similitud de documentos con Bag-Of-Words y Similitud del Coseno

## Imports

In [1]:
from pathlib import  Path
from collections import Counter

import pandas as pd
import numpy as np
import regex
from docx import Document


import nltk
from nltk.corpus import stopwords
import torch

In [2]:
nltk.download("stopwords", quiet=True)

True

## Preprocesamiento

In [3]:
route = Path("docs")
stop_es = set(stopwords.words("spanish"))

for doc in route.glob("*.docx"):
    docx = Document(doc)

    lower_doc = ""

    paragraphs = [paragraph.text.lower() for paragraph in docx.paragraphs if paragraph.text.strip()]

    lower_doc = " ".join(paragraphs)

    clean_text = regex.sub(r"[^\p{L}]+", " ", lower_doc)

    filtered_words = [word for word in clean_text.split() if word not in stop_es]

    doc_name = doc.stem
    doc_processed = Path(f"docs_processed/{doc_name}.txt")
    doc_processed.parent.mkdir(parents=True, exist_ok=True)

    doc_processed.write_text(" ".join(filtered_words).strip(), encoding="utf-8")

## Construccion de Bag-of-Words

In [4]:
route = Path("docs_processed")

documents = {}

for doc in route.glob("*.txt"):
    text = doc.read_text(encoding="utf-8")
    words = text.split()
    documents[doc.stem] = sorted(words)

vocabulary = sorted(set(word for words in documents.values() for word in words))

bow_vectors = {}

for doc_name, words in documents.items():
    words_counts = Counter(words)
    bow_vectors[doc_name] = [words_counts[word] for word in vocabulary]

In [5]:
for doc in documents.values():
    print(doc)

['además', 'alta', 'amantes', 'anunció', 'año', 'calificado', 'características', 'causarán', 'cámara', 'denominado', 'dispositivo', 'esperan', 'expertos', 'fotografía', 'gama', 'gb', 'generación', 'hoy', 'impacto', 'incorpora', 'industria', 'lanzamiento', 'marca', 'megapíxeles', 'memoria', 'mercado', 'mercado', 'meses', 'móvil', 'móvil', 'nuevo', 'nuevo', 'oled', 'pantalla', 'posee', 'procesador', 'promete', 'promete', 'próximos', 'pulgadas', 'ram', 'reconocida', 'resolución', 'revolucionar', 'revuelo', 'seguramente', 'smartphone', 'smartphone', 'sobresalientes', 'tecnophone', 'tecnophone', 'tecnophone', 'triple', 'ver', 'xz', 'xz', 'última', 'último']
['afirman', 'anunció', 'aplicaciones', 'avanzada', 'bajo', 'batería', 'batería', 'batería', 'cantidades', 'constante', 'constante', 'consumen', 'convertido', 'cumple', 'demanda', 'desafío', 'después', 'discutidos', 'duración', 'durar', 'días', 'energía', 'experiencia', 'fabricante', 'fabricantes', 'grandes', 'innovadoras', 'marcar', 'mob

In [6]:
df = pd.DataFrame(bow_vectors, index=vocabulary)
print(df.to_string())

                  doc01  doc02  doc03  doc04  doc05  doc06  doc07  doc08  doc09  doc10  doc11  doc12
actualizaciones       0      0      0      0      0      0      0      0      0      0      1      0
además                1      0      0      1      0      0      0      0      0      0      1      1
adopción              0      0      0      0      1      0      0      0      1      0      0      0
adquisición           0      0      0      0      1      0      0      0      0      0      0      0
afirman               0      1      0      0      0      0      0      0      0      0      0      0
agigantados           0      0      1      0      0      0      0      0      0      0      0      0
ahora                 0      0      0      1      0      0      0      0      0      0      0      0
alta                  1      0      0      0      0      0      0      1      0      0      0      0
amantes               1      0      0      0      0      0      0      0      0      0     

# 3. Similitud de Coseno

In [7]:
cosi = torch.nn.CosineSimilarity(dim=0)

doc_names = list(df.columns)

bow_tensors = {}

for doc_name in doc_names: 
    bow_tensors[doc_name] = torch.tensor(bow_vectors[doc_name], dtype=torch.float32)

comparaciones = []

for i in range(len(doc_names)):
    for j in range(i +1, len(doc_names)):
        doc1 = doc_names[i]
        doc2 = doc_names[j]

        similarity = cosi(bow_tensors[doc1], bow_tensors[doc2]).item()

        comparaciones.append((similarity, doc1, doc2))

        print(f"Similarity between {doc1} and {doc2}: {similarity:.4f}")


Similarity between doc01 and doc02: 0.0441
Similarity between doc01 and doc03: 0.0000
Similarity between doc01 and doc04: 0.0795
Similarity between doc01 and doc05: 0.0993
Similarity between doc01 and doc06: 0.1380
Similarity between doc01 and doc07: 0.0636
Similarity between doc01 and doc08: 0.0152
Similarity between doc01 and doc09: 0.0000
Similarity between doc01 and doc10: 0.0497
Similarity between doc01 and doc11: 0.1462
Similarity between doc01 and doc12: 0.0882
Similarity between doc02 and doc03: 0.0953
Similarity between doc02 and doc04: 0.0000
Similarity between doc02 and doc05: 0.0000
Similarity between doc02 and doc06: 0.1027
Similarity between doc02 and doc07: 0.1243
Similarity between doc02 and doc08: 0.0339
Similarity between doc02 and doc09: 0.0000
Similarity between doc02 and doc10: 0.1109
Similarity between doc02 and doc11: 0.1883
Similarity between doc02 and doc12: 0.1094
Similarity between doc03 and doc04: 0.0000
Similarity between doc03 and doc05: 0.0000
Similarity 

# 4. Análisis

In [8]:
comparaciones.sort(key=lambda x: x[0], reverse=True)    

for similarity, doc1, doc2 in comparaciones[:10]:
    print(f"Similarity between {doc1} and {doc2}: {similarity:.4f}")

Similarity between doc09 and doc12: 0.3226
Similarity between doc11 and doc12: 0.3016
Similarity between doc04 and doc12: 0.2488
Similarity between doc04 and doc09: 0.2166
Similarity between doc05 and doc09: 0.2067
Similarity between doc05 and doc08: 0.1912
Similarity between doc07 and doc11: 0.1904
Similarity between doc02 and doc11: 0.1883
Similarity between doc09 and doc11: 0.1660
Similarity between doc07 and doc12: 0.1540


In [9]:
import math

def tf_idf(term, document_words, documents):
    if not document_words:
        return 0.0

    tf = document_words.count(term) / len(document_words)

    n_documents_with_term = sum(
        term in words
        for words in documents.values()
    )

    idf = math.log(
        len(documents) / n_documents_with_term
    )

    return tf * idf


tf_idf_vectors = {}

for word in vocabulary:
    tf_idf_vectors[word] = []

    for document_words in documents.values():
        value = tf_idf(
            word,
            document_words,
            documents
        )

        tf_idf_vectors[word].append(value)

tf_idf_vectors


{'actualizaciones': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.03357981959172974,
  0.0],
 'además': [0.01894159118393293,
  0.0,
  0.0,
  0.024968461106093406,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.014846112009028512,
  0.01323629263455554],
 'adopción': [0.0,
  0.0,
  0.0,
  0.0,
  0.04266093974352512,
  0.0,
  0.0,
  0.0,
  0.03981687709395678,
  0.0,
  0.0,
  0.0],
 'adquisición': [0.0,
  0.0,
  0.0,
  0.0,
  0.059164444042571436,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'afirman': [0.0,
  0.048723659799764715,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'agigantados': [0.0,
  0.0,
  0.052870354250808514,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'ahora': [0.0,
  0.0,
  0.0,
  0.05647515113154546,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'alta': [0.030892404641863016,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.04166882486576872,
  0.0,
  0.0,
  0.0,
  0.0],
 'amant

In [10]:
doc_names = list(documents.keys())

tf_idf_df = pd.DataFrame(
    tf_idf_vectors,
    index=doc_names
)

print(tf_idf_df.round(4).to_string())

       actualizaciones  además  adopción  adquisición  afirman  agigantados   ahora    alta  amantes   ambas  amenazas  anunciado  anuncio  anunció  aplicaciones  aplicación  aporta     app    apps  aprovechar  aquellos     ar  arduamente  arworld  asequible    auge  aumentada  aumentando  aumento  autenticación  automoción  automotrices  automovilística  autonomía  autovolt  avanzada     año    bajo   banca  basándose  batería  beneficiarse  bolsillos  brilla   busca    cada    café  calidad  calificado  calles  cambiar  cambio  cantidades  capaces  características   carga  cargar  causarán  celebran  cellflex  centros   cerca  cercanas    cero  chargeongo  ciudad  ciudades  climatización  clipmaster   coche  coches  colaboración  comerciales  compacto  compatible  compañía  competitivo  compradores  comprar  comprarlos  compras  conducción  conductores  conectada  conectividad  considerados  constante  consumen  contacto  contaminación  continúa  controlar  convergiendo  convertido  